# 1. The data and the inputs

## What you will learn

- What one row of the model's table is (one annual report of one company, called a 10-K), and what the label is (an
  S&P downgrade within 12 months; S&P is a credit rating agency).
- The four groups of inputs in the main model (10-K ratios, S&P rating state, stock market, economy) and what each one
  measures. A fifth group, text tone, is only a side test.
- Why every input is "known on the day" of the report, and what goes wrong (look-ahead) if it is not.

## What you need

- The real table built by notebook 5. Run `python -m tcf_credit.rebuild data build --route free` (or
  `--route wrds --wrds-user YOUR_NAME`) from the repository folder with `PYTHONPATH=model/src` before you start.
  This notebook only reads the files that rebuild wrote.
- Python with the packages in `model/requirements.txt` (see `model/README.md`) and `jupyter`.
- On the WRDS route your table and the notebook outputs hold licensed data. Keep them on your own computer and clear
  the outputs before you share the notebook.
- Run time depends on the size of the ratings file. The notebook reads the table and the rating events and does
  no long computation; the rebuild already ran the models.


## Set up

`DATA_DIR` is where downloads and hand-downloaded files live; `OUT_DIR` is where the rebuild wrote its tables and
results. Both default to folders in the repository. The environment variables `TCF_NB_DATA_DIR` and `TCF_NB_OUT_DIR`
override them.

The next cell only sets the folders and checks for `OUT_DIR/panel_v2.csv`. If the file is missing, the notebook prints
"No real table yet: run the rebuild first (see notebook 5)." and every later cell that needs data prints one short
line and does nothing. The notebook never writes into `OUT_DIR` or `DATA_DIR`.


In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd

# Find the repository folder (the one that holds model/src/tcf_credit). This works from the repository folder or from
# model/notebooks.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "model" / "src" / "tcf_credit").is_dir())
SRC = root / "model" / "src"
sys.path[:0] = [str(SRC)]
old_path = os.environ.get("PYTHONPATH")
os.environ["PYTHONPATH"] = os.pathsep.join([str(SRC)] + ([old_path] if old_path else []))

DATA_DIR = Path(os.environ.get("TCF_NB_DATA_DIR", root / "data"))
OUT_DIR = Path(os.environ.get("TCF_NB_OUT_DIR", root / "build"))
panel_path = OUT_DIR / "panel_v2.csv"
events_path = OUT_DIR / "ratings" / "ratings_obligor_events.csv.gz"
ready = panel_path.exists()

pd.set_option("display.width", 150)
pd.set_option("display.max_columns", 40)
print("repository:", root)
print("data folder:", DATA_DIR)
print("output folder:", OUT_DIR)
if ready:
    print("Found", panel_path)
else:
    print("No real table yet: run the rebuild first (see notebook 5).")


## One row = one annual report of one company

A US company whose shares trade on a US stock exchange files an annual report with the SEC (the US securities
regulator), called a **10-K**. The model's table (the "panel") has one row for each usable 10-K of each company (a
report is left out when the company is not rated by S&P that day or is in default, or when the 12-month outcome is not
known). A company that files for ten years gives up to ten rows.

The key columns:

- `cik`: the company's number at the SEC. `sp_name`: its name in S&P's records. `accn`: the number of this one report.
- `period_end`: the balance-sheet date (such as 31 December).
- `t`: the **snapshot date**. It is the day the 10-K was filed plus one day. Everything the model knows about this row
  is what the world knew at `t`, not before and not after.
- `label`: 1 if S&P downgraded the company in the 12 months after `t`, else 0 (explained below). `rating` and `notch`:
  S&P's rating on `t`, as letters and as a number (1 = AAA, 10 = BBB-, 21 = C).

**Why the day after the filing?** A 10-K can arrive at any time of the working day of EDGAR, the SEC's filing website
(6 a.m. to 10 p.m. Eastern time, Monday to Friday), for instance while the stock market is open. If the snapshot were
stamped on the filing day itself, the model might use numbers that the market only saw later that day. One day later
we are sure the report was public. The same rule applies to prices and rating actions: only what is dated before `t`
counts.


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    panel = pd.read_csv(panel_path)
    panel["t"] = pd.to_datetime(panel["t"])
    panel["period_end"] = pd.to_datetime(panel["period_end"])
    print(panel.shape[0], "rows,", panel.shape[1], "columns,", panel.cik.nunique(), "companies")


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    display(panel[["cik", "sp_name", "period_end", "t", "label", "rating", "notch", "debt_assets",
                   "ebit_assets"]].head(6))


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    per_firm = panel.groupby("cik").size()
    print("reports per company: smallest", per_firm.min(), "| median", int(per_firm.median()), "| largest",
          per_firm.max())
    gap = (panel["t"] - panel["period_end"]).dt.days
    print("days from balance-sheet date to snapshot date t: smallest", gap.min(), "| median", int(gap.median()),
          "| largest", gap.max())


The gap in the last line comes from the filing lag. A company files its 10-K some time after its balance-sheet date.
The model may use the numbers only from `t` on. The 10-K is public on the filing date, one day before `t`; `t` adds a
day of margin.


## The label: a downgrade within 12 months

For each row we ask one question: **did S&P cut the company's long-term credit rating in the 365 days after `t`?**
The answer is the `label`: 1 = yes, 0 = no. The model's job is to give each row a probability that the label is 1.

Two rules decide the label (they are in `build_panel.py`):

1. Only a downgrade ("DG") counts. If S&P withdraws the rating first, the row is dropped, because we do not know what
   would have happened.
2. The 365 days must be over when the rating data ends. A report filed six months before the end of the data has an
   unfinished window, so its row is dropped too. That is why the table has no rows for reports filed in the last 365
   days before the end of the rating data.

The next cell uses the package's own `label` and `rating_state` functions on the real rating events of one company in
the table that has a downgrade.


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
elif not events_path.exists():
    print("No rating events yet: run the rebuild first (see notebook 5).")
else:
    import collections
    import csv
    import datetime as dt
    import gzip

    from tcf_credit.build_panel import label, rating_state

    candidates = panel.loc[panel["label"] == 1, ["obligor", "sp_name", "t", "label"]].copy()
    wanted = set(candidates["obligor"].astype(str))
    by_obligor = collections.defaultdict(list)
    data_end = None
    with gzip.open(events_path, "rt", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            day = dt.date.fromisoformat(row["action_date"][:10])
            data_end = day if data_end is None or day > data_end else data_end
            if (row["agency"] == "S&P" and row["sec_class"] == "Corporate"
                    and row["rating_type"] == "Issuer Credit Rating"
                    and row["rating_subtype"] == "Foreign Currency LT"
                    and row["obligor_id"] in wanted):
                by_obligor[row["obligor_id"]].append(
                    (day, row["rating"], row["action_class"], row["outlook"], row["watch"]))
    for events_ in by_obligor.values():
        events_.sort()
    picked, events = None, []
    for _, row in candidates.iterrows():
        found = by_obligor.get(str(row["obligor"]), [])
        if found and label(found, row["t"].date(), data_end) == 1:
            picked, events = row, found
            break
    if picked is None:
        print("No company with both a downgrade and rating events was found in OUT_DIR.")
    else:
        t = picked["t"].date()
        print("company:", picked["sp_name"], "| snapshot t:", t)
        print("rating known at t:", rating_state(events, t).get("rating"))
        print("label from these events:", label(events, t, data_end), "| label in the table:",
              int(picked["label"]))
        print("events from one year before t to one year after t:")
        for e in events:
            if t - dt.timedelta(days=365) <= e[0] <= t + dt.timedelta(days=365):
                print(" ", e[0], e[1], e[2], e[3], e[4])


**Key to the action codes** (the "action" in each event; the model's code looks only at these):

| Code | Meaning | What the model does with it |
|---|---|---|
| `DG` | downgrade | the label is 1 if one falls in the 12 months after `t`; also feeds `dg_prev_year` |
| `UP` | upgrade | feeds `up_prev_year` |
| `WD`, `WE`, `WO` | rating withdrawn (after default, paid off, other reasons) | if one comes first, the row drops |
| `NW` and any other code | `NW` is a new rating; other codes are other actions | no effect on the label |

`rating_state` uses only actions dated before `t`; `label` looks at actions dated in `(t, t+365]`. The lines printed
above come from the rating history of the company named there, so they show both rules on real events. If the rating is
withdrawn first, the row is dropped; if the window is not over by the end of the rating data, the row is dropped too.

Now the label in the real table. How many reports are followed by a downgrade, and how does that change by year?


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    panel["year"] = panel["t"].dt.year
    by_year = panel.groupby("year").label.agg(reports="size", downgrades="sum")
    by_year["share"] = (by_year.downgrades / by_year.reports).round(3)
    print("all years: ", len(panel), "reports,", int(panel.label.sum()), "downgrades,", f"{panel.label.mean():.1%}")
    display(by_year)


In the lecture's real data, the test years 2016-2024 hold 6,380 reports and 862 downgrades (13.5%). A model that gives
every company the same probability, 13.5%, is the "do nothing clever" baseline. Everything the model adds has to beat
that.


## The inputs: four groups (plus a fifth for a side test)

The model reads 43 numbers for each report. They fall into four groups. The cell below lists them with a one-line
meaning written in the notebook. The lists of inputs come from `evaluate.py` and `features_v2.py`, so if the code
changes and this table does not, the cell stops with an error.

There is a **fifth group, text tone**: the change, since the previous 10-K, in the share of negative and uncertain
words in the report (counted with the Loughran-McDonald word lists). It is used only in a side test
(`evaluate.py --text`), so it is not among the 43 inputs and not in `panel_v2.csv`.


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    from tcf_credit import evaluate as E
    from tcf_credit import features_v2 as F

    # Short meanings, in words a finance student knows.
    meaning = {
        # 10-K ratios
        "debt_assets": "debt / total assets (leverage)",
        "ebit_assets": "operating profit (EBIT) / total assets (how much the assets earn)",
        "wc_assets": "working capital (current assets - current liabilities) / assets",
        "ebit_interest": "EBIT / interest expense (interest cover)",
        "cfo_debt": "cash flow from operations / debt",
        "log_assets": "natural log of total assets (size)",
        "re_assets": "retained earnings / assets (past profits kept)",
        "cash_assets": "cash / assets",
        "op_margin": "EBIT / revenue",
        "debt_ebitda": "debt / EBITDA (set to 0 when EBITDA is negative)",
        "float_debt": "market value of shares held by outsiders / debt",
        "dividend_payer": "1 if the company paid dividends",
        "chg_debt_assets": "debt_assets minus last year's debt_assets",
        "chg_ebit_assets": "ebit_assets minus last year's",
        "chg_ebit_interest": "ebit_interest minus last year's",
        "chg_cfo_debt": "cfo_debt minus last year's",
        "chg_cash_assets": "cash_assets minus last year's",
        "chg_op_margin": "op_margin minus last year's",
        "chg_debt_ebitda": "debt_ebitda minus last year's",
        "vol_ebit_assets_3y": "spread (standard deviation) of ebit_assets over the last three reports",
        "neg_ebitda": "1 if EBITDA is negative",
        # S&P rating state
        "notch": "the rating as a number: 1 = AAA, 10 = BBB-, 21 = C",
        "ig": "1 if investment grade (BBB- or better)",
        "outlook_neg": "1 if S&P's outlook is negative",
        "outlook_pos": "1 if S&P's outlook is positive",
        "watch_neg": "1 if on CreditWatch negative (a review for a possible downgrade)",
        "watch_pos": "1 if on CreditWatch positive",
        "dg_prev_year": "1 if S&P downgraded the company in the year before t",
        "up_prev_year": "1 if S&P upgraded the company in the year before t",
        # market
        "ret_12m": "stock return over the last 12 months minus the S&P 500's",
        "ret_6m": "the same over 6 months",
        "vol_12m": "yearly volatility of the stock's daily returns",
        "mcap_debt": "market value of the shares / debt",
        "dd_naive": "distance to default (a simple Merton model: how many standard deviations the firm is "
                     "above default)",
        "beta_12m": "how strongly the stock moves with the market",
        "idio_vol_12m": "volatility of the part of the stock's return the market does not explain",
        # economy
        "unemp": "unemployment rate",
        "unemp_chg_12m": "unemployment rate minus its value a year earlier",
        "ip_growth_12m": "12-month growth of industrial production",
        "gdp_growth": "GDP growth of the latest quarter (annual rate)",
        "baa10y": "yield gap between Baa corporate bonds and 10-year Treasuries (credit spread)",
        "t10y3m": "10-year minus 3-month Treasury yield (yield curve slope)",
        "vix_20d": "average of the stock-market fear index (VIX) over 20 days",
    }
    groups = {
        "10-K ratios": E.RATIOS + E.BINARY + [c for c in F.V2_COLUMNS if c.startswith("chg_")
                                              or c in ("vol_ebit_assets_3y", "neg_ebitda")],
        "S&P rating state": E.RATING,
        "Stock market": E.MARKET + ["beta_12m", "idio_vol_12m"],
        "Economy": F.MACRO,
    }

    # The model's full list must equal our four groups. If the code changes and this table does not, this fails.
    all_inputs = [c for g in groups.values() for c in g]
    assert sorted(all_inputs) == sorted(E.RATIOS + E.RATING + E.BINARY + E.MARKET + F.V2_COLUMNS), "table out of date"
    assert set(all_inputs) <= set(panel.columns) and set(meaning) == set(all_inputs)

    rows = [(g, c, meaning[c]) for g, cols in groups.items() for c in cols]
    table = pd.DataFrame(rows, columns=["group", "input", "what it measures"])
    print(len(all_inputs), "inputs")
    display(table)


The panel has a few more columns that the model does **not** read: such as `years_rated` (how long S&P has rated the
company). It grows with calendar time, so test years look different from training years.

The model also reads the company's broad sector (a set of yes/no columns made from the industry code). Reports without
stock prices are left out of the model's test, because the market inputs are part of the model.


## Why every input is "known on the day"

Imagine you score a company on the day after it files its 10-K. You may use only what exists on that day. If a number
from the future leaks in, the model looks brilliant in a test and is useless in real life. This leak is called
**look-ahead**. Each group avoids it in a different way:

| Group | The rule that avoids look-ahead |
|---|---|
| 10-K ratios | Values **as first filed**; restatements ignored. Changes are against the previous 10-K. |
| S&P rating state | Only rating actions dated **strictly before** `t` count (a downgrade dated `t` is not used). |
| Stock market | Only trading days **before** `t`; returns, volatility and beta stop the day before `t`. |
| Economy | Numbers **as first released**; daily series use the last value before `t`; VIX is its 20-day average. |
| Text tone (side test only) | Counted in the 10-K and compared with the previous 10-K; both were public at `t`. |

The first-release unemployment and output data come from the Philadelphia Fed's real-time files. Changes in the
ratios are against the previous 10-K, filed about one year earlier (250 to 500 days).

One more guard is about testing. The label of a row looks 12 months into that row's future. So when we **test** the
model we leave a one-year gap between the last training year and the test year; otherwise a training label would
depend on events that happen after the test year has started. Notebook 2 shows this.


## A first check on the real table

The next cell checks the snapshot rule on the table you built: no snapshot date `t` comes before its balance-sheet
date. It also shows that the stock-market inputs are empty for reports with `has_prices = 0`.


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    # A simple check on the real table: no snapshot date t comes before its balance-sheet date.
    assert (panel["t"] > panel["period_end"]).all()
    # And the 12-month return and other market inputs are empty for reports without prices (has_prices = 0):
    print(panel.groupby("has_prices")[["ret_12m", "vol_12m", "dd_naive"]].count())


## The inputs against the label

Do riskier-looking companies get downgraded more often? Below, each input is cut into five equal groups (quintiles)
from low to high, and we print the downgrade rate in each group. Each `#` is one percentage point.

The tables come from the real table you built. The pattern is not put there by a rule in the notebook; it is what the
data show, and it can differ by route.


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    def rate_table(col, bins=5):
        d = panel[[col, "label"]].dropna()
        if d[col].nunique() <= 2:
            groups_ = d.groupby(col)
        else:
            groups_ = d.groupby(pd.qcut(d[col], bins, duplicates="drop"), observed=True)
        t = groups_.label.agg(reports="size", rate="mean")
        t["rate"] = t["rate"].round(3)
        t["bar"] = ["#" * int(round(r * 100)) for r in t["rate"]]
        t.index = [str(i) for i in t.index]
        t.index.name = col
        return t

    for c in ["debt_assets", "ebit_assets", "notch", "vol_12m", "baa10y"]:
        display(rate_table(c))


In [ ]:
if not ready:
    print("No real results yet: run the rebuild first (see notebook 5).")
else:
    for c in ["outlook_neg", "watch_neg", "dg_prev_year"]:
        display(rate_table(c))


Read each table from top to bottom. If the downgrade rate climbs with `debt_assets` and falls as `ebit_assets` rises,
the inputs carry a signal a credit analyst already uses; the model's work is to weigh them together. A negative
outlook or a CreditWatch placement often marks a higher rate. Your own table can differ by route, so read it before
you draw a conclusion.


**What the lecture found on the real (licensed) data, for comparison.** Test years 2016-2024, 6,380 annual
reports, of which 862 were followed by an S&P downgrade within 12 months (13.5%).

| Result (real data; the terms are taught in later notebooks) | Value |
|---|---|
| Logit: ROC-AUC / PR-AUC (both explained in notebook 2) | 0.802 / 0.494 |
| Other single engines (not the fly models): ROC-AUC | 0.785 to 0.808 (boosting 0.808; no single engine is clearly better than the logit) |
| Averages that include the logit: ROC-AUC | 0.817 (logit-boosting) and 0.813 (all non-fly models); both clearly above the logit |
| S&P rating state alone: ROC-AUC | 0.757 |
| Fly brain models: ROC-AUC (explained in notebook 4) | 0.759 to 0.807 (real wiring had no consistent edge over shuffled wiring) |
| Flag line (top 10% of an earlier year's scores; explained in notebook 3) | caught 407 of 862 downgrades, with 8.7% false alarms |


## Tidy up

This notebook only reads `OUT_DIR` and `DATA_DIR`. It writes nothing there and creates no temporary folder, so there is
nothing to delete. To build or refresh the data, use notebook 5.


## Check yourself

1. What does one row of the table stand for, and on which day is it "stamped"?

<details><summary>Answer</summary>

One annual report (10-K) of one company. It is stamped with the snapshot date `t` = the filing date plus one day.
</details>

2. A company's rating is cut on the same day that its 10-K is filed. Is that downgrade part of its rating
   state at `t`? Is it part of its label?

<details><summary>Answer</summary>

The downgrade is dated one day before `t`, so it **is** part of the rating state at `t` (the model sees the new
rating). It is not part of the label, which looks at the 365 days after `t`. A downgrade dated `t` itself is in
neither: the state uses events before `t`, the label uses events after `t`.
</details>

3. Why does the model use the first-released unemployment rate and not the final, revised one?

<details><summary>Answer</summary>

On the day of the report only the first release existed. The revised value was published later, so using it
would be look-ahead and could make test results look better than real use.
</details>
